# Metro Network w. OD analysis

In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd

import matplotlib.pyplot as plt
import matplotlib as mpl
import networkx as nx
import seaborn as sns
from itertools import tee, pairwise, combinations

In [ ]:
from scripts.network_scripts import station_flow_metrics, compute_temporal_mixing

In [ ]:
k5_palette = [
    "#265EF8",
    "#DE3EC4",
    "#F16F32", 
    "#37BB53",
    '#19180A'
]

In [ ]:
m1m2_stations = ['Flintholm', 'Lindevang', 'Fasanvej', 'Christianshavn', 'Sundby',
       'Bella Center', 'Femøren', 'Kastrup', 'Københavns Lufthavn',
       'Nørreport', 'Vanløse', 'Forum',
       'Islands Brygge', 'DR Byen', 'Ørestad', 'Vestamager', 'Amagerbro',
       'Lergravsparken', 'Øresund', 'Amager Strand'
]

m3m4_stations = ['Gammel Strand', 'Trianglen', 'Nuuks Plads',
       'Frederiksberg Allé', 'Enghave Plads', 'Havneholmen',
       'Enghave Brygge', 'Sluseholmen', 'København H',
       'Rådhuspladsen', 'Marmorkirken', 'Østerport',
       'Poul Henningsens Plads', 'Vibenshus Runddel', 'Skjolds Plads',
       'Nørrebro', 'Nørrebros Runddel', 'Aksel Møllers Have', 'Orientkaj',
       'Nordhavn', 'Mozarts Plads', 'København Syd']

m1m2_m3m4_stations = ['Kongens Nytorv', 'Frederiksberg']

In [ ]:
sns.color_palette("icefire", as_cmap=True)


In [ ]:
sns.color_palette("vlag", as_cmap=True)


In [ ]:
sns.diverging_palette(220, 20, as_cmap=True)


In [ ]:
sns.color_palette("Spectral", as_cmap=True)


In [ ]:
sns.color_palette("coolwarm", as_cmap=True)


## Loading Files

### Infrastructure & Metro Features

In [ ]:
municipalities = gpd.read_file("../data/OSM/copenhagen_metric.gpkg", layer="metro municipalities")
mun_area = gpd.read_file("../data/OSM/copenhagen_metric.gpkg", layer="greater metro area")

In [ ]:
# STEP 1: Reading in metro stations and lines
metro_stations = gpd.read_file("../data/OSM/copenhagen_metric.gpkg", layer="metro stations")
metro_lines = gpd.read_file("../data/OSM/copenhagen_metric.gpkg", layer="metro lines")

# Ensure both are in a projected CRS (meters)
gdf_stations = metro_stations.to_crs(epsg=25832)
gdf_stations.reset_index(drop=False, inplace=True)

gdf_tracks = metro_lines.to_crs(epsg=25832)
gdf_tracks.reset_index(drop=False, inplace=True)

In [ ]:
metro_lines.head()

### Neighborhood Attributes

In [ ]:
# SOCIO-DEMOGRAPHIC
social = pd.read_csv("../data/otm/processed/station_otm_norm_socio.csv")
social = social.drop(columns=['id', 'system'])
print("Social attributes:", social.shape[1]-1)

# FUNCTIONAL
functional = pd.read_csv("../data/processed/station_functional.csv")
functional = functional.drop(columns=['id', 'system'])
print("Functional attributes:", functional.shape[1]-1)

# ENVIRONMENT
env = pd.read_csv("../data/processed/station_landcomposition_basic.csv")
env = env.drop(columns=['id', 'system'])
print("Environment attributes:", env.shape[1]-1)

# ALL COMBINED
all = pd.read_csv("../data/processed/station_characteristics.csv")
all = all.drop(columns=['id', 'system'])
print("All Neighborhood Attributes:", all.shape[1]-1)

In [ ]:
all.head()

### Neighborhood Clusters

In [ ]:
# FINAL
neigh_types = pd.read_csv("../results/clustering/neighborhood/multi_method_labels.csv")

### Station pairs & Distance

In [ ]:
station_pairs = pd.read_excel("../data/Metroselskabet/station_pairs_distance.xlsx", sheet_name="Sheet2")
station_labels = pd.read_excel("../data/Metroselskabet/stopPoint2Name.xlsx")
station_list = station_labels["stationName"].unique().tolist()

# Create mapping dictionary from abbreviations to full names
mapping = dict(zip(station_labels["stopName"], station_labels["stationName"]))

# Map onto df_edges
station_pairs["Fra_full"] = station_pairs["Fra"].map(mapping)
station_pairs["Til_full"] = station_pairs["Til"].map(mapping)
station_pairs.head()

### OD Flows

In [ ]:
# reading in OD values
od_total = pd.read_csv("../data/Metroselskabet/OD/total/OD_total.csv")
#df_od_cluster = assign cluster value for each station...
#od_cluster_gb = df_od_cluster.groupby(['from_cluster', 'to_cluster']).sum()

print(od_total.shape)
od_total.head()

In [ ]:
od_morning = pd.read_csv("../data/Metroselskabet/OD/mean/OD_morning_peak.csv")
od_noon = pd.read_csv("../data/Metroselskabet/OD/mean/OD_midday.csv")
od_afternoon = pd.read_csv("../data/Metroselskabet/OD/mean/OD_afternoon_peak.csv")
od_evening = pd.read_csv("../data/Metroselskabet/OD/mean/OD_evening.csv")
od_night = pd.read_csv("../data/Metroselskabet/OD/mean/OD_night.csv")

print(od_morning.shape)
od_morning.head()

print(od_afternoon.shape)
od_afternoon.head()

In [ ]:
od_all = od_total.merge(od_morning, on=['from', 'to'],
                   how='left')
od_all = od_all.merge(od_noon, on=['from', 'to'],
                   how='left')
od_all = od_all.merge(od_afternoon, on=['from', 'to'],
                   how='left')
od_all = od_all.merge(od_evening, on=['from', 'to'],
                   how='left')
od_all = od_all.merge(od_night, on=['from', 'to'],
                   how='left')
od_all = od_all.rename(columns={"morning_peak_trips": "morning_trips",
                                "midday_trips": "noon_trips",
                                "afternoon_peak_trips": "afternoon_trips"})
od_all.head()

In [ ]:
od_all.columns

# Graph Construction

### Base Graph (Distance weighted)

In [ ]:
# Base graph, edges weighted by distance
BG = nx.Graph()
for index, row in station_pairs.iterrows():
    u = row["Fra_full"]
    v = row["Til_full"]
    d = row["Afstand (meter)"]
    #t = row['Travel time (seconds)']
    #print(u, v, d)
    BG.add_edge(u, v, distance=d) # travel_time=t

# TO DO: Add node attributes from gdf_stations    

print("Undirected topology graph built.")
print("Nodes:", BG.number_of_nodes())
print("Edges:", BG.number_of_edges())

In [ ]:
nx.draw(BG)

In [ ]:
# Get station coordinates
gdf_stations_indexed = gdf_stations.set_index("name")
nodes = list(BG.nodes())
gdf_ordered = gdf_stations_indexed.reindex(nodes)

station_coordinates = np.column_stack((
    gdf_ordered.geometry.x.values,
    gdf_ordered.geometry.y.values
))

# To plot with networkx, we need to merge the nodes back to
# their positions in order to plot in networkx
positions = dict(zip(BG.nodes, station_coordinates))

#f, ax = plt.subplots(1, 2, figsize=(8, 4))
nx.draw(BG, positions, node_size=5, node_color="blue")
plt.show()

In [ ]:
# TO DO: Save graph as gpkg?
#https://networkx.org/documentation/latest/auto_examples/geospatial/plot_osmnx.html

### Directed and Flow Weighted Grap

In [ ]:
DG = nx.DiGraph()

def pairwise(iterable):
    a, b = tee(iterable)
    next(b, None)
    return zip(a, b)

# Add edges in both directions (From base graph)
for u, v, data in BG.edges(data=True):
    dist = data["distance"]
    DG.add_edge(u, v, distance=dist, 
                flow=0.0,
                flow_morning=0.0,
                flow_noon=0.0,
                flow_afternoon=0.0,
                flow_evening=0.0,
                flow_night=0.0
                )
    DG.add_edge(v, u, distance=dist, 
                flow=0.0,
                flow_morning=0.0,
                flow_noon=0.0,
                flow_afternoon=0.0,
                flow_evening=0.0,
                flow_night=0.0
                )

print("\nDirected graph created for OD flow assignment.")
print("Directed edges:", DG.number_of_edges())

In [ ]:
# ASSIGN OD FLOWS TO SHORTEST PATHS

missing_paths = 0

for _, row in od_all.iterrows():
    O = row['from']        # adjust column names as needed
    D = row['to']
    F = row['trips']          # hourly average passengers
    F_morning = row['morning_trips']
    F_noon = row['noon_trips']
    F_afternoon = row['afternoon_trips']
    F_evening = row['evening_trips']
    F_night = row['night_trips']

    # Skip cases where O == D
    if O == D:
        continue

    try:
        # Compute shortest path by distance
        path = nx.shortest_path(DG, O, D, weight="distance")

        # Assign flow to each directed edge in the path
        for u, v in pairwise(path):
            DG[u][v]["flow"] += F
            DG[u][v]["flow_morning"] += F_morning
            DG[u][v]["flow_noon"] += F_noon
            DG[u][v]["flow_afternoon"] += F_afternoon
            DG[u][v]["flow_evening"] += F_evening
            DG[u][v]["flow_night"] += F_night
            

    except nx.NetworkXNoPath:
        missing_paths += 1
        continue

for u, v, d in DG.edges(data=True):
    d["inv_flow"] = 1 / d["flow"] # inverted flow

print("\nOD Flow assignment completed.")
print("Missing OD paths:", missing_paths)

In [ ]:
# --- Extract flow values for all edges ---
flows = np.array([data["flow"] for _, _, data in DG.edges(data=True)])

# Avoid vmin=vmax (happens if all flows are zero)
vmin = flows.min()
vmax = flows.max() if flows.max() > flows.min() else flows.min() + 1e-6

cmap = plt.cm.plasma
fig, ax = plt.subplots(figsize=(10, 10))
ax.set_axis_off()

# Plot nodes
nx.draw_networkx_nodes(
    DG, positions,
    node_size=10, node_color="indigo",
    ax=ax)

# Plot edges with colormap based on flow
edge_coll = nx.draw_networkx_edges(
    DG, positions,
    edge_color=flows,         # <-- IMPORTANT
    edge_cmap=cmap,
    edge_vmin=vmin, edge_vmax=vmax,
    width=2, arrows=False,             # optional, arrows make LineCollection tricky
    ax=ax)

# Add colorbar
cbar = plt.colorbar(edge_coll, ax=ax)
cbar.set_label("Flow", rotation=90)
plt.show()

In [ ]:
flows = np.array([data["flow_morning"] for _, _, data in DG.edges(data=True)])

# Avoid vmin=vmax (happens if all flows are zero)
vmin = flows.min()
vmax = flows.max() if flows.max() > flows.min() else flows.min() + 1e-6

cmap = plt.cm.plasma
fig, ax = plt.subplots(figsize=(10, 10))
ax.set_axis_off()

# Plot nodes
nx.draw_networkx_nodes(
    DG, positions,
    node_size=30, node_color="indigo",
    ax=ax)

# Plot edges with colormap based on flow
edge_coll = nx.draw_networkx_edges(
    DG, positions,
    edge_color=flows,         # <-- IMPORTANT
    edge_cmap=cmap,
    edge_vmin=vmin, edge_vmax=vmax,
    width=8, arrows=False,             # optional, arrows make LineCollection tricky
    ax=ax)

# Add colorbar
cbar = plt.colorbar(edge_coll, ax=ax)
cbar.set_label("Hourly Morning Flow", rotation=90)
plt.show()

In [ ]:
# --- Extract flow values for all edges ---
flows = np.array([data["flow_afternoon"] for _, _, data in DG.edges(data=True)])

# Avoid vmin=vmax (happens if all flows are zero)
vmin = flows.min()
vmax = flows.max() if flows.max() > flows.min() else flows.min() + 1e-6

cmap = plt.cm.plasma
fig, ax = plt.subplots(figsize=(10, 10))
ax.set_axis_off()

# Plot nodes
nx.draw_networkx_nodes(
    DG, positions,
    node_size=30, node_color="indigo",
    ax=ax)

# Plot edges with colormap based on flow
edge_coll = nx.draw_networkx_edges(
    DG, positions,
    edge_color=flows,         # <-- IMPORTANT
    edge_cmap=cmap,
    edge_vmin=vmin, edge_vmax=vmax,
    width=8, arrows=False,             # optional, arrows make LineCollection tricky
    ax=ax)

# Add colorbar
cbar = plt.colorbar(edge_coll, ax=ax)
cbar.set_label("Hourly Afternoon Flow", rotation=90)
plt.show()

# Distances

In [ ]:
# Precompute all-pairs shortest path lengths
# Result: dict-of-dicts: dist[u][v]
all_distances = dict(nx.all_pairs_dijkstra_path_length(BG, weight="distance"))

In [ ]:
def compute_within_cluster_distance(stations_df, cluster_col, dist_dict):
    """
    stations_df : DataFrame with columns ['name', cluster_col]
    cluster_col : str, e.g. 'k_5'
    dist_dict   : output of all_pairs_dijkstra_path_length
    
    Returns
    -------
    DataFrame with per-cluster distance statistics
    """

    results = []

    for cluster_id, group in stations_df.groupby(cluster_col):
        stations = group["name"].tolist()

        # Skip trivial clusters
        if len(stations) < 2:
            continue

        dists = []
        for i, j in combinations(stations, 2):
            if i in dist_dict and j in dist_dict[i]:
                dists.append(dist_dict[i][j])

        results.append({
            "cluster": cluster_id,
            "n_stations": len(stations),
            "mean_within_dist": np.mean(dists),
            "median_within_dist": np.median(dists),
            "min_within_dist": np.min(dists),
            "max_within_dist": np.max(dists)
        })

    return pd.DataFrame(results)


In [ ]:
within_dist_df = compute_within_cluster_distance(
    stations_df=neigh_types,
    cluster_col="k_5",
    dist_dict=all_distances
)

within_dist_df


# Flow Metrics

In [ ]:
def station_flow_metrics(DiG,
                         BaG, 
                         od_df, 
                         trips_cols=['trips', 'morning_trips', 'noon_trips', 'afternoon_trips', 'evening_trips', 'night_trips'], 
                         #flow_col=["flow", "flow_morning", "flow_noon", "flow_afternoon", "flow_evening", "flow_night"],
                         temporal_norm=False, 
                         station_names=None):
    metrics_df = pd.DataFrame()

    for trip_col in trips_cols:
        if trip_col =="trips":
            flow_col = "flow"
        else:
            time_split = trip_col.split("_")[0]
            flow_col = "flow_" + time_split
        print(trip_col, flow_col)

        # 1. start and end nodes
        origins = od_df.groupby("from")[trip_col].sum().reindex(DG.nodes(), fill_value=0)
        destinations = od_df.groupby("to")[trip_col].sum().reindex(DG.nodes(), fill_value=0)
        metrics_df[f'{trip_col}_origins'] = origins
        metrics_df[f'{trip_col}_destinations'] = destinations

        # 2. node inflow and outflow
        inflow = DiG.in_degree(weight=flow_col)     # People arriving at or through station
        outflow = DiG.out_degree(weight=flow_col)   # People departing at or through station
        inflow_values = {node: inflow[node] for node in DiG.nodes()}
        outflow_values = {node: outflow[node] for node in DiG.nodes()}
        metrics_df[f'{trip_col}_inflow'] = inflow_values
        metrics_df[f'{trip_col}_outflow'] = outflow_values

        # 3. node through-flow
        node_flow = {node: 0 for node in DiG.nodes()}
        for u, v, data in DiG.edges(data=True):
            node_flow[u] += data[flow_col]
            node_flow[v] += data[flow_col]
        metrics_df[f'{trip_col}_node_flow'] = node_flow

        # 3. Transfer Flow
        through_flow = {}
        for n in DiG.nodes():
            through_flow[n] = inflow[n] + outflow[n] - origins[n] - destinations[n]
        metrics_df[f'{trip_col}_through_flow'] = through_flow


    # 4. structural centralities
    closeness = nx.closeness_centrality(BaG, distance="distance")
    betweenness = nx.betweenness_centrality(BaG, weight="distance")
    metrics_df['closeness'] = closeness
    metrics_df['betweenness'] = betweenness

    if temporal_norm:
        EPS = 1e-6

        # identify base (daily) columns
        base_outflow = metrics_df["trips_outflow"]
        base_node_flow = metrics_df["trips_node_flow"]

        for trip_col in trips_cols:
            if trip_col == "trips":
                continue  # skip base, used for normalization

            inflow = metrics_df[f"{trip_col}_inflow"]
            outflow = metrics_df[f"{trip_col}_outflow"]
            node_flow = metrics_df[f"{trip_col}_node_flow"]
            through_flow = metrics_df[f"{trip_col}_through_flow"]

            # --- Directional imbalance ---
            metrics_df[f"{trip_col}_net_flow"] = outflow - inflow
            metrics_df[f"{trip_col}_net_flow_norm"] = (
                metrics_df[f"{trip_col}_net_flow"] / (node_flow + EPS)
            )

            metrics_df[f"{trip_col}_out_in_ratio"] = (
                outflow / (inflow + EPS)
            )

            # --- Temporal share (within-station normalization) ---
            metrics_df[f"{trip_col}_outflow_share"] = (
                outflow / (base_outflow + EPS)
            )

            metrics_df[f"{trip_col}_node_flow_share"] = (
                node_flow / (base_node_flow + EPS)
            )

            # --- Transfer intensity ---
            metrics_df[f"{trip_col}_through_flow_share"] = (
                through_flow / (node_flow + EPS)
            )



    metrics_df = metrics_df.reset_index(names=['station'])
    return metrics_df

In [ ]:
all_flow_metrics = station_flow_metrics(DG, BG, od_all)
all_flow_metrics.head()

In [ ]:
all_flow_metrics2 = station_flow_metrics(DG, BG, od_all, temporal_norm=True)
all_flow_metrics2.head()

In [ ]:
normalization_map = {
    "morning_trips_inflow": "trips_inflow",
    "morning_trips_outflow": "trips_outflow",
    "morning_trips_node_flow": "trips_node_flow",
    "morning_trips_through_flow": "trips_through_flow",

    "evening_trips_inflow": "trips_inflow",
    "evening_trips_outflow": "trips_outflow",
    "evening_trips_node_flow": "trips_node_flow",
    "evening_trips_through_flow": "trips_through_flow",

    "night_trips_inflow": "trips_inflow",
    "night_trips_outflow": "trips_outflow",
    "night_trips_node_flow": "trips_node_flow",
    "night_trips_through_flow": "trips_through_flow",
}


df_rel = all_flow_metrics.copy()

for num_col, denom_col in normalization_map.items():
    df_rel[num_col + "_share"] = (
        df_rel[num_col] / df_rel[denom_col]
    )

df_rel.replace([float("inf"), -float("inf")], pd.NA, inplace=True)

df_rel[
    ["morning_trips_inflow_share",
     "evening_trips_inflow_share",
     "night_trips_inflow_share"]
].sum(axis=1).describe()



In [ ]:
df_rel[[
    "station",
    "trips_inflow",
    "morning_trips_inflow",
    "morning_trips_inflow_share"
]]


#### Neighborhood correlation

In [ ]:
flow_neighatt = all_flow_metrics.merge(all,
                                       left_on='station',
                                       right_on='name',
                                       how='left')


In [ ]:
flow_cols = [i for i in all_flow_metrics.columns[1:]]
flow_cols

In [ ]:
flow_corr_cols = ['trips_origins', 'trips_destinations',
                  'trips_inflow', 'trips_outflow',
                  'trips_node_flow', 'trips_through_flow',
                  'closeness', 'betweenness']
neigh_att_cols = ['POP', 'POP15_S', 'HOUSEH1', 'HOUSEH3', 
                  'JOBS', 'JOBS_POP15', 'STU_POP',
                  'CARS', 'CDEN', 'INCLOW_S', 'INCHIGH_S',
                  'B_TOTAL', 'B_RES', 'GREEN', 'POI_TOTAL'
]

corr = flow_neighatt[flow_corr_cols + neigh_att_cols].corr()
corr_flow_neigh = corr.loc[flow_corr_cols, neigh_att_cols]
corr_flow_neigh

In [ ]:
corr_flow_neigh = (
    flow_neighatt[flow_corr_cols + neigh_att_cols]
    .corr(method="spearman")
    .loc[flow_corr_cols, neigh_att_cols]
)

corr_flow_neigh

In [ ]:
strong = (
    corr_flow_neigh
    .abs()
    .stack()
    .sort_values(ascending=False)
)

strong[strong > 0.5]


In [ ]:
x = flow_neighatt["B_TOTAL"]
y = flow_neighatt["trips_origins"]

plt.figure(figsize=(4,4))
plt.scatter(x, y, alpha=0.7)
plt.xlabel("Jobs")
plt.ylabel("Closeness")
plt.title("Buildings vs Trips Origin")

# optional trend line
m, b = np.polyfit(x, y, 1)
plt.plot(x, m*x + b)

plt.show()

In [ ]:
x = flow_neighatt["B_TOTAL"]
y = flow_neighatt["trips_destinations"]

plt.figure(figsize=(4,4))
plt.scatter(x, y, alpha=0.7)
plt.xlabel("Jobs")
plt.ylabel("Closeness")
plt.title("Buildings vs Trips Origin")

# optional trend line
m, b = np.polyfit(x, y, 1)
plt.plot(x, m*x + b)

plt.show()

In [ ]:
x = flow_neighatt["JOBS"]
y = flow_neighatt["closeness"]

plt.figure(figsize=(4,4))
plt.scatter(x, y, alpha=0.7)
plt.xlabel("Jobs")
plt.ylabel("Closeness")
plt.title("Closeness vs Jobs")

# optional trend line
m, b = np.polyfit(x, y, 1)
plt.plot(x, m*x + b)

plt.show()


In [ ]:
plt.scatter(x.rank(), y.rank())
plt.xlabel("Jobs (rank)")
plt.ylabel("Closeness (rank)")
plt.title("Rank–rank relationship")
plt.show()


In [ ]:
metrics = ["trips_origins", "trips_node_flow", "closeness"]

fig, axes = plt.subplots(1, len(metrics), figsize=(12,4))

for ax, m in zip(axes, metrics):
    ax.scatter(flow_neighatt["JOBS"], flow_neighatt[m], alpha=0.7)
    ax.set_title(m)
    ax.set_xlabel("Jobs")

plt.tight_layout()
plt.show()


# OD Analysis

## Observed OD Mixing
P_obs tells something about::
- Diagonal dominance → spatial / functional self-containment
- Strong off-diagonal entries → inter-cluster coupling
- Asymmetry → hierarchy (e.g., residential → work ≠ work → residential)
- Low row entropy → specialized origin clusters
- High row entropy → diversified mobility patterns

In [ ]:
# Function to compute mixing for a single time slice
def compute_temporal_mixing(od, trip_col):
    """
    od: dataframe with columns cluster_from, cluster_to, trip_col
    trip_col: str, e.g. 'morning_trips'
    """

    df = (
        od
        .groupby(["cluster_from", "cluster_to"], as_index=False)[trip_col]
        .sum()
        .rename(columns={trip_col: "trips"})
    )

    df["total_out"] = df.groupby("cluster_from")["trips"].transform("sum")
    df["P_obs"] = (df["trips"] / df["total_out"]) * 100

    P = df.pivot(
        index="cluster_from",
        columns="cluster_to",
        values="P_obs"
    ).fillna(0)

    # absolute volume
    F_obs = (
        df
        .pivot(index="cluster_from", columns="cluster_to", values="trips")
        .fillna(0)
    )

    return P

In [ ]:
od_focus = od_all[["from", "to", "morning_trips", "afternoon_trips"]].copy()

chosen_k = 'k_5' # CLUSTER RESOLUTION
stations_df = neigh_types.copy()
stations_df = stations_df[['name', chosen_k]]

od_focus = od_focus.merge(
    stations_df.rename(columns={"name": "from", chosen_k: "cluster_from"}),
    on="from",
    how="left"
    )

od_focus = od_focus.merge(
    stations_df.rename(columns={"name": "to", chosen_k: "cluster_to"}),
    on="to",
    how="left"
    )
od_focus.head()

In [ ]:
# total trips departing clusters
od_fromcluster_gb = od_focus.groupby(['cluster_from'])[['morning_trips', 'afternoon_trips']].sum()
od_fromcluster_gb

# total trips arriving clusters
od_tocluster_gb = od_focus.groupby(['cluster_to'])[['morning_trips', 'afternoon_trips']].sum()
od_tocluster_gb

#### Cluster-level (heatmaps/matrices)

In [ ]:
P_morning = compute_temporal_mixing(od_focus, "morning_trips")
P_afternoon = compute_temporal_mixing(od_focus, "afternoon_trips")

# enforce cluster order
clusters = sorted(stations_df[chosen_k].unique())
P_morning = P_morning.reindex(index=clusters, columns=clusters, fill_value=0)
P_afternoon = P_afternoon.reindex(index=clusters, columns=clusters, fill_value=0)

P_temporal = P_afternoon - P_morning

In [ ]:
P_morning

In [ ]:
P_afternoon

In [ ]:
fig, (ax1, ax2) = plt.subplots(nrows=1, ncols=2, figsize=(16, 7),
                               gridspec_kw={"wspace": 0.2}
                               )
ax1 = sns.heatmap(P_morning, cmap='Greys', ax=ax1, annot=True, fmt=".0f",
                  vmin=10, vmax=45, cbar=False,
                  annot_kws={"size": 12})
ax2 = sns.heatmap(P_afternoon, cmap='Greys', ax=ax2, annot=True, fmt=".0f",
                  vmin=10, vmax=45, cbar=False,
                  annot_kws={"size": 12})

#ax1.set(xlabel="To Cluster", ylabel="From Cluster")
#ax2.set(xlabel="To Cluster", ylabel="From Cluster")

ax1.set_title('Morning Peak', fontsize=18)
ax2.set_title('Afternoon Peak', fontsize=18)

for ax in (ax1, ax2):
    ax.set_xlabel("To Cluster", fontsize=14)
    ax.set_ylabel("From Cluster", fontsize=14)
    ax.tick_params(axis='both', labelsize=12)

plt.tight_layout()
plt.savefig("../results/plots/od/od_heatmap.png")
plt.show()

In [ ]:
fig, (ax1, ax2) = plt.subplots(
    1, 2,
    figsize=(17, 8),
    gridspec_kw={"wspace": 0.15}
)

# Create an axis for the shared colorbar
cbar_ax = fig.add_axes([0.92, 0.15, 0.02, 0.7])
# [left, bottom, width, height] in figure coordinates

sns.heatmap(
    P_morning,
    cmap='Greys',
    ax=ax1, annot=True, fmt=".0f",
    vmin=10, vmax=45, cbar=True, cbar_ax=cbar_ax
)

sns.heatmap(
    P_afternoon,
    cmap='Greys',
    ax=ax2, annot=True, fmt=".0f",
    vmin=10, vmax=45, cbar=False
)

ax1.set_title("Avg. Morning Peak Hour Flow", fontsize=18)
ax2.set_title("Avg. Afternoon Peak Hour Flow", fontsize=18)

for ax in (ax1, ax2):
    ax.set_xlabel("To Cluster", fontsize=14)
    ax.set_ylabel("From Cluster", fontsize=14)
    ax.tick_params(axis='both', labelsize=12)

plt.tight_layout()
plt.savefig("../results/plots/od/od_heatmap_v2.png")
plt.show()

In [ ]:
fig, (ax1, ax2) = plt.subplots(
    1, 2,
    figsize=(17, 8),
    gridspec_kw={"wspace": 0.15}
)

# Create an axis for the shared colorbar
cbar_ax = fig.add_axes([0.92, 0.15, 0.02, 0.7])
# [left, bottom, width, height] in figure coordinates

sns.heatmap(
    P_morning,
    cmap='Greys',
    ax=ax1,
    annot=True,
    fmt=".0f",
    annot_kws={"size": 14},   # 👈 increase this
    vmin=10, vmax=45,
    cbar=True, cbar_ax=cbar_ax
)

sns.heatmap(
    P_afternoon,
    cmap='Greys',
    ax=ax2, annot=True, fmt=".0f",
    annot_kws={"size": 14},   # 👈 increase this
    vmin=10, vmax=45, cbar=False
)

ax1.set_title("Avg. Morning Peak Hour Flow", fontsize=18)
ax2.set_title("Avg. Afternoon Peak Hour Flow", fontsize=18)

for ax in (ax1, ax2):
    ax.set_xlabel("To Cluster", fontsize=14)
    ax.set_ylabel("From Cluster", fontsize=14)

    for tick, color in zip(ax.get_xticklabels(), k5_palette):
        tick.set_color(color)
        tick.set_fontsize(14)

    for tick, color in zip(ax.get_yticklabels(), k5_palette):
        tick.set_color(color)
        tick.set_fontsize(14)
    
cbar_ax.tick_params(labelsize=14)  # increase this value

plt.tight_layout()
plt.savefig("../results/plots/od/od_heatmap_v2.png")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))

sns.heatmap(
    P_temporal,
    cmap='coolwarm',
    center=0,
    vmin=-8, vmax=8,   # scale to your actual probabilities
    ax=ax,
    annot=True, fmt=".0f",
    annot_kws={"size": 14}
)

#ax.set(
#    xlabel="To Cluster",
#    ylabel="From Cluster",
#    title="Change in OD mixing probabilities (Afternoon − Morning)"
#)

ax.set_title("Change in Peak Flow (Afternoon - Morning)", fontsize=18)

ax.set_xlabel("To Cluster", fontsize=14)
ax.set_ylabel("From Cluster", fontsize=14)
ax.tick_params(axis='both', labelsize=14)
ax.collections[0].colorbar.ax.tick_params(labelsize=14)

for tick, color in zip(ax.get_yticklabels(), k5_palette):
    tick.set_color(color)
    tick.set_fontsize(14)

for tick, color in zip(ax.get_xticklabels(), k5_palette):
    tick.set_color(color)
    tick.set_fontsize(14)

plt.tight_layout()
plt.savefig("../results/plots/od/od_heatmap_norm.png")
plt.show()

In [ ]:
within_dist_df["P_morning_cc"] = within_dist_df["cluster"].map(
    lambda c: P_morning.loc[c, c]
)

within_dist_df["P_afternoon_cc"] = within_dist_df["cluster"].map(
    lambda c: P_afternoon.loc[c, c]
)

# OD Flow Network Metrics

Make a DataFrame like:<br>
station	inflow	outflow	through_flow	closeness	betweenness	cluster

This lets you see:
- some stations are structurally central but not heavily used
- others are heavily used but not central (crowded suburbs)

In [ ]:
def station_flow_metrics(DiG,
                         BaG, 
                         od_df, 
                         trips_cols=['trips', 'morning_trips', 'noon_trips', 'afternoon_trips', 'evening_trips', 'night_trips'], 
                         #flow_col=["flow", "flow_morning", "flow_noon", "flow_afternoon", "flow_evening", "flow_night"],
                         station_names=None):
    metrics_df = pd.DataFrame()

    for trip_col in trips_cols:
        if trip_col =="trips":
            flow_col = "flow"
        else:
            time_split = trip_col.split("_")[0]
            flow_col = "flow_" + time_split
        print(trip_col, flow_col)

        # 1. start and end nodes
        origins = od_df.groupby("from")[trip_col].sum().reindex(DG.nodes(), fill_value=0)
        destinations = od_df.groupby("to")[trip_col].sum().reindex(DG.nodes(), fill_value=0)
        metrics_df[f'{trip_col}_origins'] = origins
        metrics_df[f'{trip_col}_destinations'] = destinations

        # 2. node inflow and outflow
        inflow = DiG.in_degree(weight=flow_col)     # People arriving at or through station
        outflow = DiG.out_degree(weight=flow_col)   # People departing at or through station
        inflow_values = {node: inflow[node] for node in DiG.nodes()}
        outflow_values = {node: outflow[node] for node in DiG.nodes()}
        metrics_df[f'{trip_col}_inflow'] = inflow_values
        metrics_df[f'{trip_col}_outflow'] = outflow_values

        # 3. node through-flow
        node_flow = {node: 0 for node in DiG.nodes()}
        for u, v, data in DiG.edges(data=True):
            node_flow[u] += data[flow_col]
            node_flow[v] += data[flow_col]
        metrics_df[f'{trip_col}_node_flow'] = node_flow

        # 3. transfer flow
        through_flow = {}
        for n in DiG.nodes():
            through_flow[n] = inflow[n] + outflow[n] - origins[n] - destinations[n]
        metrics_df[f'{trip_col}_through_flow'] = through_flow


    # 4. structural centralities
    closeness = nx.closeness_centrality(BaG, distance="distance")
    betweenness = nx.betweenness_centrality(BaG, weight="distance")
    metrics_df['closeness'] = closeness
    metrics_df['betweenness'] = betweenness

    metrics_df = metrics_df.reset_index(names=['station'])
    return metrics_df

In [ ]:
station_flow_metrics = station_flow_metrics(DG, BG, od_all)
station_flow_metrics.head()

In [ ]:
# normalize at station level
station_flow_metrics['closeness_norm'] = station_flow_metrics['closeness'] / all_flow_metrics['closeness'].mean()
station_flow_metrics['betweenness_norm'] = station_flow_metrics['betweenness'] / all_flow_metrics['betweenness'].mean()

In [ ]:
station_flow_metrics_w_types_all = station_flow_metrics.merge(neigh_types,
                                           left_on='station', right_on='name',
                                           how='left')
station_flow_metrics_w_types_all.head()
station_flow_metrics_w_types_all.to_csv("../results_flow_metrics_all.csv", index=False)

In [ ]:
cluster_col = "k_5" # CLUSTER RESOLUTION
df = station_flow_metrics_w_types_all.rename(columns={cluster_col: "cluster"}).copy()

In [ ]:
# station temporal shares from node_flow per time period
periods = ["morning", "noon", "afternoon", "evening", "night"]
node_flow_cols = {p: f"{p}_trips_node_flow" for p in periods}

# Safety check: make sure expected columns exist
missing = [c for c in node_flow_cols.values() if c not in df.columns]
if missing:
    raise ValueError(
        "Missing expected node-flow columns:\n"
        + "\n".join(missing)
        + "\n\nCheck your CSV column names (e.g., 'noon' vs 'midday')."
    )

# total station activity
df["total_activity"] = df[list(node_flow_cols.values())].sum(axis=1)

# station-level shares
for p, col in node_flow_cols.items():
    df[f"share_{p}"] = np.where(df["total_activity"] > 0, df[col] / df["total_activity"], 0.0)

In [ ]:
# station-level highlight off-peak tendencies

# peak share = morning + afternoon
df["peak_share"] = df["share_morning"] + df["share_afternoon"]
df["offpeak_share"] = 1.0 - df["peak_share"]  # noon + evening + night

# inflow_asymmetry: 1 = morning sink, -1 = afternoon sink, 0 = balanced
if {"morning_trips_inflow", "afternoon_trips_inflow"}.issubset(df.columns):
    den = df["morning_trips_inflow"] + df["afternoon_trips_inflow"]
    df["inflow_asymmetry"] = np.where(
        den > 0,
        (df["morning_trips_inflow"] - df["afternoon_trips_inflow"]) / den,
        0.0
    )
else:
    df["inflow_asymmetry"] = np.nan  # if those columns don’t exist in your file

In [ ]:
# cluster-level aggregation: mean + spread, IQR
def iqr(x):
    x = pd.to_numeric(x, errors="coerce")
    return np.nanpercentile(x, 75) - np.nanpercentile(x, 25)

cluster_summary = (
    df.groupby("cluster")
      .agg(
          n_stations=("station", "count") if "station" in df.columns else ("cluster", "size"),
          peak_share_mean=("peak_share", "mean"),
          peak_share_iqr=("peak_share", iqr),
          offpeak_share_mean=("offpeak_share", "mean"),
          offpeak_share_iqr=("offpeak_share", iqr),
          inflow_asym_mean=("inflow_asymmetry", "mean"),
          inflow_asym_iqr=("inflow_asymmetry", iqr),
      )
      .reset_index()
      .sort_values("cluster")
)

print("\nCluster summary (temporal profile indicators):")
print(cluster_summary)

In [ ]:
# identify "atypical" stations within each cluster (salience within-cluster)
# station is atypical if (|z| > 1).

threshold = 1
indicators = ["offpeak_share", "inflow_asymmetry"]

atyp = df.loc[:, [c for c in ["station", "cluster"] + indicators if c in df.columns]].copy()

for ind in indicators:
    if ind not in atyp.columns:
        continue
    mu = atyp.groupby("cluster")[ind].transform("mean")
    sd = atyp.groupby("cluster")[ind].transform("std").replace(0, np.nan)
    atyp[f"z_{ind}_within_cluster"] = (atyp[ind] - mu) / sd
    atyp[f"atypical_{ind}"] = atyp[f"z_{ind}_within_cluster"].abs() > threshold

# Count atypical stations per cluster
atyp_counts = (
    atyp.groupby("cluster")
        .agg(
            atypical_offpeak=("atypical_offpeak_share", "sum") if "atypical_offpeak_share" in atyp.columns else ("cluster", "size"),
            atypical_asym=("atypical_inflow_asymmetry", "sum") if "atypical_inflow_asymmetry" in atyp.columns else ("cluster", "size"),
        )
        .reset_index()
)

# If some atypical columns weren't created, fix counts gracefully
if "atypical_offpeak_share" not in atyp.columns:
    atyp_counts["atypical_offpeak"] = np.nan
if "atypical_inflow_asymmetry" not in atyp.columns:
    atyp_counts["atypical_asym"] = np.nan

cluster_summary = cluster_summary.merge(
    atyp_counts[["cluster", "atypical_offpeak", "atypical_asym"]],
    on="cluster",
    how="left"
)

print("\nCluster summary + atypical counts:")
print(cluster_summary)

In [ ]:
# most off-peak-oriented stations
if "station" in df.columns:
    top_offpeak = (
        df.sort_values("offpeak_share", ascending=False)
          .loc[:, [c for c in ["station", "cluster", "offpeak_share", "inflow_asymmetry", "trips_node_flow"] if c in df.columns]]
          .head(10)
    )
    print("Top 10 off-peak-oriented stations:")
    print(top_offpeak)

In [ ]:
cluster_shares = (
    df.groupby("cluster")[[f"share_{p}" for p in periods]]
      .mean()
      .sort_index()
)

fig, ax = plt.subplots(figsize=(9, 4.8))
bottom = np.zeros(len(cluster_shares))
x = np.arange(len(cluster_shares.index))

for p in periods:
    vals = cluster_shares[f"share_{p}"].values
    ax.bar(x, vals, bottom=bottom, label=p.capitalize())
    bottom += vals

ax.set_xticks(x)
ax.set_xticklabels([f"Cluster {c}" for c in cluster_shares.index])
ax.set_ylabel("Mean share of station activity")
ax.set_title("Cluster temporal signatures (mean station activity share by time period)")
ax.legend(ncol=5, bbox_to_anchor=(0.5, -0.12), loc="upper center", frameon=False)
ax.set_ylim(0, 1)

plt.tight_layout()
#plt.savefig(out_path, dpi=200)
plt.show()

#### Selected columns...

In [ ]:
flow_cols = ['closeness', 'betweenness',
             'closeness_norm', 'betweenness_norm',
             'trips_origins', 'trips_destinations', 'trips_inflow', 'trips_outflow', 'trips_node_flow', 'trips_through_flow',
             'morning_trips_origins', 'morning_trips_destinations',
             'morning_trips_inflow', 'morning_trips_outflow',
             'morning_trips_node_flow', 'morning_trips_through_flow',
             'afternoon_trips_origins', 'afternoon_trips_destinations',
             'afternoon_trips_inflow', 'afternoon_trips_outflow',
             'afternoon_trips_node_flow', 'afternoon_trips_through_flow']

station_flow_metrics_w_types = station_flow_metrics_w_types_all[['name', 'k_5'] + flow_cols]

station_flow_metrics_w_types.to_csv("../results_flow_metrics.csv", index=False)

In [ ]:
cluster_means = all_flow_metrics2.groupby('k_5')[flow_cols].mean()
cluster_means.reset_index()

In [ ]:
df_table_example.T.reset_index()

In [ ]:
cluster_means.T.reset_index()

In [ ]:
# print latex table example 
df_table_example = cluster_means.T.reset_index()
print(df_table_example.to_latex(index=False,
                  formatters={"name": str.upper},
                  float_format="{:.2f}".format,
))  

In [ ]:
cluster_means_save = cluster_means.reset_index()
cluster_means_save.to_csv("../results/network_metrics.csv", index=False)

### Correlation

In [ ]:
base_cols = ['closeness', 'betweenness']
flow_cols = []
base_corr_cols = ['']

In [ ]:
flow_corr_cols = ['JOBS_POP15', 'B_TOTAL', 'B_RES', 'POP']

In [ ]:
flow_cols = [i for i in all_flow_metrics.columns[1:]]
social_cols = [i for i in social.columns[1:]]
flow_neighatt = all_flow_metrics.merge(all,
                                       left_on='station',
                                       right_on='name',
                                       how='left')

flow_neighatt.head()

In [ ]:
all.columns

In [ ]:
corr = flow_neighatt[flow_cols + social_cols].corr()

# keep only flow vs neighborhood block
corr_flow_neigh = corr.loc[flow_cols, social_cols]
corr_flow_neigh

In [ ]:
corr_flow_neigh = (
    flow_neighatt[flow_cols + social_cols]
    .corr(method="spearman")
    .loc[flow_cols, social_cols]
)

corr_flow_neigh

In [ ]:
strong = (
    corr_flow_neigh
    .abs()
    .stack()
    .sort_values(ascending=False)
)

strong[strong > 0.5]


In [ ]:
for i in strong:
    print(strong)

In [ ]:
corr_long = (
    corr_flow_neigh
    .reset_index()
    .melt(id_vars="index",
          var_name="neighborhood_var",
          value_name="correlation")
    .rename(columns={"index": "flow_var"})
)
corr_long.sort_values(by="correlation").head(20)

In [ ]:
sns.regplot(data=flow_neighatt, x="through_flow", y="HOUSEH3", ci=None)
#sns.regplot(data=test2, x="closeness", y="ONOP", ci=None)

# Notes
Using shortest path assignment flows to compute in-degree/out-degree does NOT give actual arrival/departure volumes.

To compute:
“Where do people come from?” -> Use origin_flow.
“Where do people go?” ->Use destination_flow.

To compute:
“Which stations are important because many people pass through?” -> Use through_flow.

To compute:
“Which edges carry the most passenger load?” -> Use link_flow.

To compute:
“Which stations are structurally important in the network regardless of flows?” -> Use graph centrality (betweenness, closeness) on distance, not flows.

# Archived

In [ ]:
test = metrics_df.reset_index(names=['station'])
test # merge with neighborhood types
#test.groupby('k_5').sum()

In [ ]:
def station_flow_metrics(DiG,
                         BaG, 
                         od_df, 
                         trips_col="trips", 
                         flow_col="flow",
                         station_names=None):
    metrics_df = pd.DataFrame()

    # 1. start and end nodes
    origins = od_df.groupby("from")[trips_col].sum().reindex(DG.nodes(), fill_value=0)
    destinations = od_df.groupby("to")[trips_col].sum().reindex(DG.nodes(), fill_value=0)
    metrics_df['origins'] = origins
    metrics_df['destinations'] = destinations

    # 2. node inflow and outflow
    inflow = DiG.in_degree(weight=flow_col)     # People arriving at or through station
    outflow = DiG.out_degree(weight=flow_col)   # People departing at or through station
    inflow_values = {node: inflow[node] for node in DiG.nodes()}
    outflow_values = {node: outflow[node] for node in DiG.nodes()}
    metrics_df['inflow'] = inflow_values
    metrics_df['outflow'] = outflow_values

    # 3. node through-flow
    node_flow = {node: 0 for node in DG.nodes()}
    for u, v, data in DG.edges(data=True):
        node_flow[u] += data[flow_col]
        node_flow[v] += data[flow_col]
    metrics_df['node_flow'] = node_flow

    # 3. Transfer Flow
    through_flow = {}
    for n in DiG.nodes():
        through_flow[n] = inflow[n] + outflow[n] - origins[n] - destinations[n]
    metrics_df['through_flow'] = through_flow


    # 4. Structural centralities
    closeness = nx.closeness_centrality(BaG, distance="distance")
    betweenness = nx.betweenness_centrality(BaG, weight="distance")
    metrics_df['closeness'] = closeness
    metrics_df['betweenness'] = betweenness

    metrics_df = metrics_df.reset_index(names=['station'])
    return metrics_df

In [ ]:
morning_flow_metrics = station_flow_metrics(DG, BG, 
                                            od_all,
                                            trips_col="morning_peak_trips",
                                            flow_col="flow_morning")
morning_flow_metrics

In [ ]:
test = station_flow_metrics(DG, BG, od_total)
flow_cols = [i for i in test.columns[1:]]
social_cols = [i for i in social.columns[1:]]
test2 = test.merge(all,
                   left_on='station',
                   right_on='name',
                   how='left')

test2.head()